In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


# MetricExplorer — Simple Demo

Synthetic SSB aberration sweep: 3 bf_radius groups × 11 defocus values = 33 points.

Hover any chart point to preview the phase image. Click to select.

In [2]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# Generate realistic-looking synthetic phase images
# Simulate a crystal lattice that gets sharper near optimal defocus
H, W = 128, 128
y, x = torch.meshgrid(
    torch.linspace(0, 8 * np.pi, H, device=device),
    torch.linspace(0, 8 * np.pi, W, device=device),
    indexing="ij",
)

# Base crystal lattice pattern
lattice = torch.sin(x) * torch.sin(y) + 0.5 * torch.sin(2 * x + y)

# Defocus values and bf_radius groups
defocus_values = np.linspace(-50, 50, 11)
bf_radii = [20, 25, 30]

points = []
for r in bf_radii:
    for c10 in defocus_values:
        # Simulate defocus blur: further from zero = more blur
        sigma = 1.0 + 0.08 * abs(c10) + 0.01 * abs(r - 25)
        kernel_size = max(3, int(sigma * 4) | 1)
        
        # Gaussian blur via convolution
        k = torch.arange(kernel_size, device=device, dtype=torch.float32) - kernel_size // 2
        kernel_1d = torch.exp(-0.5 * (k / sigma) ** 2)
        kernel_1d = kernel_1d / kernel_1d.sum()
        
        blurred = lattice.clone()
        # Separable convolution
        blurred = torch.nn.functional.conv2d(
            blurred.unsqueeze(0).unsqueeze(0),
            kernel_1d.view(1, 1, -1, 1),
            padding=(0, 0),
        )
        blurred = torch.nn.functional.conv2d(
            blurred,
            kernel_1d.view(1, 1, 1, -1),
            padding=(0, 0),
        )
        phase = blurred.squeeze().cpu().numpy()
        
        # Pad back to original size
        pad_h = H - phase.shape[0]
        pad_w = W - phase.shape[1]
        phase = np.pad(phase, ((pad_h // 2, pad_h - pad_h // 2), (pad_w // 2, pad_w - pad_w // 2)))
        
        # Add noise scaled by defocus
        noise = np.random.randn(H, W).astype(np.float32) * (0.05 + 0.002 * abs(c10))
        phase = phase + noise
        
        # Compute metrics
        variance_loss = float(np.var(phase)) 
        tv = float(np.sum(np.abs(np.diff(phase, axis=0))) + np.sum(np.abs(np.diff(phase, axis=1))))
        
        # FFT sharpness: ratio of high-freq to low-freq power
        fft_mag = np.abs(np.fft.fftshift(np.fft.fft2(phase)))
        center = np.array(fft_mag.shape) // 2
        mask_lo = np.zeros_like(fft_mag, dtype=bool)
        mask_lo[center[0]-5:center[0]+5, center[1]-5:center[1]+5] = True
        fft_snr = float(fft_mag[~mask_lo].mean() / (fft_mag[mask_lo].mean() + 1e-10))
        
        points.append({
            "image": phase.astype(np.float32),
            "label": f"r={r} C10={c10:+.0f}nm",
            "metrics": {
                "variance_loss": variance_loss,
                "tv": tv,
                "fft_snr": fft_snr,
            },
            "params": {
                "C10_nm": c10,
                "bf_radius": r,
            },
        })

print(f"Generated {len(points)} points, {H}x{W} images")

Using: cuda
Generated 33 points, 128x128 images


In [ ]:
from quantem.widget import MetricExplorer

explorer = MetricExplorer(
    points,
    x_key="C10_nm",
    x_label="Defocus C10 (nm)",
    group_key="bf_radius",
    metric_labels={"tv": "Total Variation", "fft_snr": "FFT SNR"},
    metric_directions={"variance_loss": "max", "tv": "min", "fft_snr": "max"},
)
explorer

In [5]:
# Read back the selected point
explorer.summary()

ShowMetricExplorer
Points:   33
Image:    128 x 128
Metrics:  variance_loss, tv, fft_snr
X axis:   Defocus C10 (nm)
Group by: bf_radius
Refs:     BF, DPC
Selected: [0] r=20 C10=-50nm
  variance_loss: 0.05491
  tv: 5673
  fft_snr: 0.1617
Display:  inferno


In [6]:
# After clicking a point in the widget, read selected params
print("Selected index:", explorer.selected_index)
print("Selected params:", explorer.selected_params)
print("Selected image shape:", explorer.selected_image.shape)

Selected index: 0
Selected params: {'C10_nm': np.float64(-50.0), 'bf_radius': 20}
Selected image shape: (128, 128)
